[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/niccoloridi/International-Law-and-AI/blob/main/IntlLaw_AI_Lab06_Pigeon_Vision.ipynb)

In [ ]:
#@title Lab 06 — Pigeon Vision
print("""
 █████             █████    █████                                                     █████████   █████
░░███             ░░███    ░░███                                         ███         ███░░░░░███ ░░███ 
 ░███  ████████   ███████   ░███         ██████   █████ ███ █████       ░███        ░███    ░███  ░███ 
 ░███ ░░███░░███ ░░░███░    ░███        ░░░░░███ ░░███ ░███░░███     ███████████    ░███████████  ░███ 
 ░███  ░███ ░███   ░███     ░███         ███████  ░███ ░███ ░███    ░░░░░███░░░     ░███░░░░░███  ░███ 
 ░███  ░███ ░███   ░███ ███ ░███      █ ███░░███  ░░███████████         ░███        ░███    ░███  ░███ 
 █████ ████ █████  ░░█████  ███████████░░████████  ░░████░████          ░░░         █████   █████ █████
░░░░░ ░░░░ ░░░░░    ░░░░░  ░░░░░░░░░░░  ░░░░░░░░    ░░░░ ░░░░                      ░░░░░   ░░░░░ ░░░░░ 
                                                                                                       
                                                                                                       
                                                                                                       

   Lab 06 // Pigeon Vision
   Melbourne Law Masters 2026
""")

*Brought to you by **Dr Niccolò Ridi**, Purveyor of Fine Scripts* — [KCL Profile](https://www.kcl.ac.uk/people/niccolo-ridi)

# Lab 06: Pigeon Vision — Object Detection and Autonomous Targeting

## Overview

In this lab, you will use a pre-trained YOLO object-detection model to:

1. **Classify and locate objects** in images and video
2. **Adjust confidence thresholds** and observe how bounding boxes, class labels, and probability scores change
3. **Understand the targeting pipeline** from pixels to decision — connecting computer vision to International Humanitarian Law
4. **Analyze error modes** (false positives, false negatives) and their human cost
5. **Reflect on meaningful human control** in autonomous systems

### Context: Lavender and the Targeting Pipeline

In 2024, investigative reporting revealed that the Israeli military used a machine learning system called **Lavender** to generate targeting recommendations in Gaza. According to these reports:

- The system scored individuals on a scale (likelihood of being a militant)
- A threshold was set: above it → recommend for airstrike
- A human operator had ~20 seconds to approve or reject each recommendation
- The volume of targets was such that meaningful individual review was reportedly impossible

This lab makes that pipeline **concrete and interactive**. You will not build a real targeting system—you will understand the logic of one, and grapple with the ethical and legal tensions embedded in that logic.

### What You'll Learn

- **Technical**: How YOLO works; what confidence scores mean; how thresholds affect detection
- **Legal**: IHL principles of distinction, proportionality, and precaution; the CCW debate on autonomous weapons
- **Ethical**: The difference between detection and targeting; the role of human judgment; error and civilian harm

# Table of Contents

> ## Part One: What Does a Machine "See"?
> [Jump to Part One](#scrollTo=part-one)

> ## Part Two: Confidence and Thresholds — When Is a Person a "Target"?
> [Jump to Part Two](#scrollTo=part-two)

> ## Part Three: From Classification to Targeting — The Lavender Pipeline
> [Jump to Part Three](#scrollTo=part-three)

> ## Part Four: Error Analysis — False Positives Kill People
> [Jump to Part Four](#scrollTo=part-four)

> ## Part Five: Meaningful Human Control — Reflection
> [Jump to Part Five](#scrollTo=part-five)

> ## Review Quiz
> [Jump to Quiz](#scrollTo=quiz)

# Part One: What Does a Machine "See"?

#@markdown
## How Does Computer Vision Work?

A digital image is not a picture. It is a matrix of numbers.

- **Pixels**: Each pixel has red, green, and blue (RGB) values, typically 0-255
- **Tensor**: An image is a 3D tensor: height × width × 3 (for RGB)
- **Convolutional Neural Networks (CNNs)**: Deep learning models with layers of filters that detect patterns
  - Early layers detect edges, corners, textures
  - Middle layers detect shapes (wheels, ears, faces)
  - Late layers detect objects (cars, animals, people)

**YOLO** (You Only Look Once) is a real-time object detection architecture:
- Divides an image into a grid
- For each grid cell, predicts: *Is there an object here? What class? How confident?*
- Outputs bounding boxes (x, y, width, height) and confidence scores (0.0 to 1.0)

**Key insight**: The model doesn't "understand" what it sees. It assigns probability scores to regions. Those scores are learned from labeled data. *A confidence of 0.95 means the model pattern-matched to the training data, not that the object is definitely there.*

In [ ]:
#@title ## Install and Import YOLO + Dependencies
!pip install -q ultralytics opencv-python-headless matplotlib numpy pillow requests

from ultralytics import YOLO
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import urllib.request
import os
from IPython.display import Image as IPImage, display
import warnings
warnings.filterwarnings('ignore')

print("YOLO and dependencies installed successfully.")

In [ ]:
#@title ## Load YOLOv8 Nano Model
#@markdown We use the nano model (yolov8n.pt) for speed on Colab free tier. 
#@markdown This model is trained on COCO dataset (80 classes: person, car, dog, etc.)

# Load the pre-trained YOLOv8 nano model
model = YOLO('yolov8n.pt')

# COCO class labels
coco_classes = {
    0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle', 4: 'airplane',
    5: 'bus', 6: 'train', 7: 'truck', 8: 'boat', 9: 'traffic light',
    10: 'fire hydrant', 11: 'stop sign', 12: 'parking meter', 13: 'bench',
    14: 'cat', 15: 'dog', 16: 'horse', 17: 'sheep', 18: 'cow', 19: 'elephant',
    20: 'bear', 21: 'zebra', 22: 'giraffe', 23: 'backpack', 24: 'umbrella',
    25: 'handbag', 26: 'tie', 27: 'suitcase', 28: 'frisbee', 29: 'skis',
    30: 'snowboard', 31: 'sports ball', 32: 'kite', 33: 'baseball bat',
    34: 'baseball glove', 35: 'skateboard', 36: 'surfboard', 37: 'tennis racket',
    38: 'bottle', 39: 'wine glass', 40: 'cup', 41: 'fork', 42: 'knife',
    43: 'spoon', 44: 'bowl', 45: 'banana', 46: 'apple', 47: 'sandwich',
    48: 'orange', 49: 'broccoli', 50: 'carrot', 51: 'hot dog', 52: 'pizza',
    53: 'donut', 54: 'cake', 55: 'chair', 56: 'couch', 57: 'potted plant',
    58: 'bed', 59: 'dining table', 60: 'toilet', 61: 'tv', 62: 'laptop',
    63: 'mouse', 64: 'remote', 65: 'keyboard', 66: 'microwave', 67: 'oven',
    68: 'toaster', 69: 'sink', 70: 'refrigerator', 71: 'book', 72: 'clock',
    73: 'vase', 74: 'scissors', 75: 'teddy bear', 76: 'hair drier', 77: 'toothbrush'
}

print(f"Model loaded: {model.model_name}")
print(f"Classes: {len(coco_classes)} (COCO dataset)")
print(f"Key classes for this lab: {coco_classes[0]}, {coco_classes[2]}, {coco_classes[5]}")

In [ ]:
#@title ## Download a Test Image: Crowded Street Scene
#@markdown We'll use a public domain image from Wikimedia Commons: a busy street scene.

# Download a public domain street scene image from Wikimedia
url = "https://upload.wikimedia.org/wikipedia/commons/7/73/Street_scene_in_Hong_Kong.jpg"
img_path = "/tmp/street_scene.jpg"

try:
    urllib.request.urlretrieve(url, img_path)
    print(f"Image downloaded: {img_path}")
except:
    # Fallback: use a simpler public domain image
    url = "https://upload.wikimedia.org/wikipedia/commons/thumb/e/eb/Ash_Tree-Dimensional_Neighbors.jpg/1024px-Ash_Tree-Dimensional_Neighbors.jpg"
    urllib.request.urlretrieve(url, img_path)
    print(f"Image downloaded (fallback): {img_path}")

# Display the image
img = Image.open(img_path)
print(f"Image size: {img.size}")
print(f"Image shape (as tensor): {np.array(img).shape}")
print(f"Pixel value range: 0-255 (typical for RGB images)")
display(img.resize((600, 400)))

In [ ]:
#@title ## Show Raw Pixel Values
#@markdown This cell displays what the image actually is: a matrix of numbers.

img_array = np.array(Image.open(img_path))
print(f"Image as NumPy array shape: {img_array.shape}")
print(f"Data type: {img_array.dtype}")
print()
print("Sample pixel values (top-left 5x5, first channel):")
print(img_array[:5, :5, 0])
print()
print("Sample pixel values (top-left 5x5, all three channels [RGB]):")
print(img_array[:5, :5, :])
print()
print("The machine does not 'see' a street. It sees a matrix of numbers.")
print("Each number is a pixel intensity (0=dark, 255=bright).")

In [ ]:
#@title ## Run YOLO Detection
#@markdown Detect objects in the image using YOLOv8 nano. Default confidence threshold: 0.25

# Run inference with default confidence threshold (0.25)
results = model.predict(img_path, conf=0.25, verbose=False)

# Extract detections
detections = results[0].boxes

print(f"Detections found: {len(detections)}")
print()
print("First 10 detections (class, confidence, bounding box):")
for i, box in enumerate(detections[:10]):
    class_id = int(box.cls[0].item())
    confidence = box.conf[0].item()
    bbox = box.xyxy[0].tolist()
    class_name = coco_classes.get(class_id, f"Unknown ({class_id})")
    print(f"  {i+1}. {class_name:15} | conf: {confidence:.3f} | box: {[f'{v:.0f}' for v in bbox]}")

In [ ]:
#@title ## Visualize Detections with Bounding Boxes
#@markdown Display the image with YOLO bounding boxes, class labels, and confidence scores.

def draw_detections(img_path, results, title="YOLO Detections"):
    """Draw bounding boxes and labels on image."""
    img = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    detections = results[0].boxes
    
    for box in detections:
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
        
        class_id = int(box.cls[0].item())
        confidence = box.conf[0].item()
        class_name = coco_classes.get(class_id, f"cls_{class_id}")
        
        # Draw bounding box
        cv2.rectangle(img_rgb, (x1, y1), (x2, y2), (0, 255, 0), 2)
        
        # Draw label
        label = f"{class_name} {confidence:.2f}"
        cv2.putText(img_rgb, label, (x1, y1 - 10), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
    
    return img_rgb

# Draw and display
img_with_boxes = draw_detections(img_path, results, "Detections at conf=0.25")

fig, ax = plt.subplots(figsize=(12, 8))
ax.imshow(img_with_boxes)
ax.set_title("YOLO Detections: Objects and Classes", fontsize=14, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.show()

print(f"\nVisualization complete. Total objects detected: {len(detections)}")

## Key Insight from Part One

A machine learning model does not "see" in the way humans see. It:

1. Takes a matrix of pixel values as input
2. Passes it through layers of learned filters
3. Outputs: bounding box coordinates and a **confidence score** (0.0 to 1.0)

The confidence score is **not** a certainty. It is the model's estimate of how likely the pattern it detected matches the patterns in its training data.

**For object detection**: A confidence of 0.95 means the model is 95% confident based on its training. But training data has biases, gaps, and errors. And the real world is different from the training data.

**For targeting**: A score of 0.95 does not mean the person is definitely a combatant. Yet military systems often treat high scores as sufficient for lethal action. This is where law, ethics, and machine learning collide.

# Part Two: Confidence and Thresholds — When Is a Person a "Target"?

#@markdown
## Understanding Confidence Thresholds

In YOLO, a **threshold** is a minimum confidence score. Detections below the threshold are discarded.

- **Low threshold (0.1)**: Many detections, including weak ones → high false positive rate (things that aren't there, or misidentified)
- **Medium threshold (0.25)**: Balanced (YOLO default)
- **High threshold (0.8)**: Few detections, only the strongest matches → high false negative rate (things we miss)

**The critical question**: *At what confidence level does a pixel pattern become a "person"? At what confidence level does a "person" become a "target"?*

This is not a philosophical question. This is a parameter setting. And parameter settings have life-or-death consequences.

In [ ]:
#@title ## Threshold Comparison: Multiple Confidence Levels
#@markdown Run detection at different confidence thresholds: 0.1, 0.25, 0.5, 0.8

thresholds = [0.1, 0.25, 0.5, 0.8]
results_by_threshold = {}
counts_by_threshold = {}
person_counts = {}

for threshold in thresholds:
    results = model.predict(img_path, conf=threshold, verbose=False)
    detections = results[0].boxes
    
    results_by_threshold[threshold] = results
    counts_by_threshold[threshold] = len(detections)
    
    # Count person-class detections
    person_count = sum(1 for box in detections if int(box.cls[0].item()) == 0)
    person_counts[threshold] = person_count

print("Detection counts by threshold:")
print("-" * 60)
for threshold in thresholds:
    total = counts_by_threshold[threshold]
    persons = person_counts[threshold]
    print(f"  conf >= {threshold:3.2f}: {total:3d} objects detected | {persons:3d} persons")
print("-" * 60)
print()
print("Key observation: Lowering the threshold increases detections.")
print("Some of these new detections are real; some are false positives.")

In [ ]:
#@title ## Visualize Threshold Effects (2x2 Grid)
#@markdown Display detections side-by-side at different thresholds.

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

for idx, threshold in enumerate(thresholds):
    results = results_by_threshold[threshold]
    img_with_boxes = draw_detections(img_path, results, f"conf >= {threshold}")
    
    axes[idx].imshow(img_with_boxes)
    axes[idx].set_title(
        f"Threshold = {threshold} | {counts_by_threshold[threshold]} detections | {person_counts[threshold]} persons",
        fontsize=11, fontweight='bold'
    )
    axes[idx].axis('off')

plt.suptitle("How Confidence Threshold Changes What Gets Detected", 
             fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

print("\nObservation: As threshold increases, fewer objects are flagged.")
print("At threshold=0.8, we miss some real objects (false negatives).")
print("At threshold=0.1, we flag things that aren't there (false positives).")

In [ ]:
#@title ## Interactive Threshold Adjustment
#@markdown Adjust the slider to see how detection changes in real-time.

from ipywidgets import FloatSlider, Output, interact
import ipywidgets as widgets

def interactive_threshold(confidence_threshold):
    """Interactive detection with user-specified threshold."""
    results = model.predict(img_path, conf=confidence_threshold, verbose=False)
    detections = results[0].boxes
    
    img_with_boxes = draw_detections(img_path, results, f"conf >= {confidence_threshold}")
    
    fig, ax = plt.subplots(figsize=(12, 8))
    ax.imshow(img_with_boxes)
    
    person_count = sum(1 for box in detections if int(box.cls[0].item()) == 0)
    vehicle_count = sum(1 for box in detections if int(box.cls[0].item()) in [2, 3, 5, 6, 7, 8])
    
    title = f"Threshold = {confidence_threshold:.2f} | Total: {len(detections)} | Persons: {person_count} | Vehicles: {vehicle_count}"
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.axis('off')
    plt.tight_layout()
    plt.show()

slider = FloatSlider(
    value=0.25,
    min=0.05,
    max=0.95,
    step=0.05,
    description='Confidence:',
    layout=widgets.Layout(width='50%')
)

interact(interactive_threshold, confidence_threshold=slider)

print("\nAdjust the slider to see how detections change.")
print("Notice: Higher threshold → fewer detections")
print("Notice: Lower threshold → more detections (including potential false positives)")

## Key Insight from Part Two

The confidence threshold is a **policy knob**. Turning it changes the behavior of the system:

| Threshold | Effect | Risk |
|-----------|--------|------|
| Low (0.1) | Detects more objects | False positives (innocent people marked for targeting) |
| Medium (0.25) | Balanced | Some misses, some false alarms |
| High (0.8) | Detects only the most obvious | False negatives (combatants missed) |

**In military operations**:
- False positives mean civilians targeted (war crimes)
- False negatives mean combatants missed (military failure)

The International Humanitarian Law principle of **DISTINCTION** requires combatants to do everything feasible to minimize false positives. The principle of **MILITARY NECESSITY** creates pressure to minimize false negatives.

**The threshold is where this tension is resolved — not by lawyers, but by engineers.**

# Part Three: From Classification to Targeting — The Lavender Pipeline

#@markdown
## The Lavender System Conceptually

According to investigative reports, the Israeli military system **Lavender** operated as follows:

1. **Detection**: Find person-shaped objects in surveillance video
2. **Re-identification**: Track the same person over time and space
3. **Scoring**: Assign a threat score (1-100 scale) based on:
   - Association with known militants (network analysis)
   - Behavior (location, movements, timing)
   - Other machine learning signals
4. **Thresholding**: If score > threshold → recommend for airstrike
5. **Human review**: Operator has ~20 seconds to approve or reject
   - At scale, meaningful review is reportedly impossible
6. **Execution**: Strike authorized

In this lab, we simulate a **simplified version** of steps 1-4 using YOLO + a synthetic threat score. We do NOT actually target anyone. We demonstrate the **logic** of the pipeline.

**Critical note**: Our threat scores are **random**. A real system would use intelligence data, network analysis, and behavioral signals. The randomness here is a placeholder to illustrate the concept.

In [ ]:
#@title ## Step 1: Detect Persons
#@markdown Use YOLO to find all person-class objects.

# Detect at lower threshold to find all potential persons
results = model.predict(img_path, conf=0.15, verbose=False)
detections = results[0].boxes

# Filter for person class (class_id=0)
person_detections = [box for box in detections if int(box.cls[0].item()) == 0]

print(f"Step 1: Detection")
print(f"Total objects: {len(detections)}")
print(f"Person objects: {len(person_detections)}")
print()
print("Person detections (class_id=0):")
for i, box in enumerate(person_detections[:15]):
    confidence = box.conf[0].item()
    bbox = box.xyxy[0].tolist()
    x1, y1, x2, y2 = bbox
    width = x2 - x1
    height = y2 - y1
    print(f"  Person {i+1}: conf={confidence:.3f}, bbox=({x1:.0f}, {y1:.0f}), size={width:.0f}x{height:.0f}")

In [ ]:
#@title ## Step 2: Assign Random Threat Scores
#@markdown Each detected person gets a simulated threat score (0-100).
#@markdown In a real system, this would come from intelligence analysis.

np.random.seed(42)  # Reproducible randomness

# Assign threat scores (0-100) to each person
threat_scores = np.random.randint(10, 95, size=len(person_detections))

print(f"Step 2: Threat Scoring")
print(f"Assigned synthetic threat scores (0-100) to {len(person_detections)} persons.")
print()
print("Person | Threat Score")
print("-" * 25)
for i, score in enumerate(threat_scores):
    print(f"  {i+1:2d}   |    {score:3d}")
print("-" * 25)
print()
print("⚠️  IMPORTANT: These are RANDOM scores for demonstration only.")
print("    A real targeting system would use intelligence analysis,")
print("    behavioral data, network analysis, and other signals.")
print("    The randomness here illustrates the pipeline, not actual targeting logic.")

In [ ]:
#@title ## Step 3: Apply Threat Threshold
#@markdown Set a threshold. Persons with threat_score > threshold are "flagged" for targeting.

threat_threshold = 60  # Adjustable

print(f"Step 3: Thresholding")
print(f"Threat threshold: {threat_threshold}")
print()

flagged = []
unflagged = []
for i, (person_box, score) in enumerate(zip(person_detections, threat_scores)):
    if score > threat_threshold:
        flagged.append((i, person_box, score))
    else:
        unflagged.append((i, person_box, score))

print(f"Flagged for targeting: {len(flagged)}")
for idx, box, score in flagged:
    print(f"  Person {idx+1}: score={score} (> {threat_threshold}) ✓ FLAGGED")

print()
print(f"Not flagged: {len(unflagged)}")
for idx, box, score in unflagged[:5]:  # Show first 5
    print(f"  Person {idx+1}: score={score} (<= {threat_threshold})")
if len(unflagged) > 5:
    print(f"  ... and {len(unflagged) - 5} more")

print()
print(f"Result: {len(flagged)} targeting recommendations generated.")

In [ ]:
#@title ## Visualize Targeting Recommendations
#@markdown Flagged persons: RED boxes | Not flagged: GREEN boxes

def draw_targeting_pipeline(img_path, person_detections, threat_scores, threat_threshold):
    """Draw bounding boxes colored by threat threshold.
    Red = flagged for targeting (threat > threshold)
    Green = not flagged (threat <= threshold)
    """
    img = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    for person_box, score in zip(person_detections, threat_scores):
        x1, y1, x2, y2 = person_box.xyxy[0].tolist()
        x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
        
        # Color: red if flagged, green if not
        if score > threat_threshold:
            color = (255, 0, 0)  # Red: flagged for targeting
            label = f"FLAGGED: {score}"
        else:
            color = (0, 255, 0)  # Green: not flagged
            label = f"safe: {score}"
        
        cv2.rectangle(img_rgb, (x1, y1), (x2, y2), color, 2)
        cv2.putText(img_rgb, label, (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
    
    return img_rgb

img_targeting = draw_targeting_pipeline(img_path, person_detections, threat_scores, threat_threshold)

fig, ax = plt.subplots(figsize=(14, 10))
ax.imshow(img_targeting)
ax.set_title(
    f"Targeting Pipeline: Threat Threshold = {threat_threshold}\n"
    f"RED (flagged) = {len(flagged)} | GREEN (not flagged) = {len(unflagged)}",
    fontsize=13, fontweight='bold'
)
ax.axis('off')
plt.tight_layout()
plt.show()

print(f"\nVisualization: RED boxes = targeting recommendations | GREEN boxes = not flagged")

In [ ]:
#@title ## Interactive Threat Threshold
#@markdown Adjust the threat threshold slider to see how many people get flagged.

def interactive_targeting(threat_threshold):
    """Interactive targeting pipeline with user-controlled threshold."""
    flagged_count = sum(1 for score in threat_scores if score > threat_threshold)
    unflagged_count = len(threat_scores) - flagged_count
    
    img_targeting = draw_targeting_pipeline(img_path, person_detections, threat_scores, threat_threshold)
    
    fig, ax = plt.subplots(figsize=(14, 10))
    ax.imshow(img_targeting)
    ax.set_title(
        f"Threat Threshold = {threat_threshold} | Flagged: {flagged_count} | Not flagged: {unflagged_count}",
        fontsize=13, fontweight='bold'
    )
    ax.axis('off')
    plt.tight_layout()
    plt.show()
    
    print(f"Threshold = {threat_threshold}")
    print(f"  Flagged (targeting recommendations): {flagged_count}")
    print(f"  Not flagged: {unflagged_count}")
    if flagged_count > 0:
        print(f"  Flagged persons: {[s for s in threat_scores if s > threat_threshold]}")

slider_threat = FloatSlider(
    value=60,
    min=10,
    max=90,
    step=5,
    description='Threshold:',
    layout=widgets.Layout(width='50%')
)

interact(interactive_targeting, threat_threshold=slider_threat)

print("\n⚠️  OBSERVE: Lowering the threshold flags more people.")
print("    These are not all guilty. Some are false positives.")
print("    But the system recommends them for targeting anyway.")

## Key Insight from Part Three

You just built a simplified version of an autonomous targeting pipeline:

1. **Detect** objects (people) using computer vision
2. **Score** them using learned models (threat/militant likelihood)
3. **Threshold** the scores (if score > X, recommend for action)
4. **Authorize** (human operator approves/rejects)

The differences between this lab and Lavender are:
- **Scale**: Lavender processed thousands of people per day
- **Training data**: Lavender used real intelligence data (we used random scores)
- **Time pressure**: Lavender operators had ~20 seconds per decision (we have unlimited time)
- **Consequences**: Lavender targeting led to airstrikes and death (our exercise does not)

But the **logic is identical**. And the problems are the same:
- Scoring systems are imperfect
- Thresholds are arbitrary
- False positives = innocent civilians targeted
- At scale, human review becomes impossible

# Part Four: Error Analysis — False Positives Kill People

#@markdown
## Why Detection Fails

Machine learning models make two types of errors:

- **False Positive (FP)**: Model says "person" but there is no person
- **False Negative (FN)**: Model says "not a person" but there is a person

In targeting systems:
- **FP = innocent person targeted** (humanitarian catastrophe, war crime)
- **FN = combatant missed** (military failure)

Detection fails because:
1. Objects are partially occluded (hidden by buildings, vehicles, etc.)
2. Objects are in unusual poses or contexts
3. Lighting is poor or unusual
4. The object is visually similar to something else in training data
5. Training data doesn't cover this scenario

We'll test YOLO on deliberately tricky images to see failure modes.

In [ ]:
#@title ## Download Test Images: Normal vs. Tricky Scenarios
#@markdown Get a mannequin, a crowd, a partially hidden person, and a normal scene.

# Dictionary of test images: label -> URL
test_images = {
    "normal_person": "https://upload.wikimedia.org/wikipedia/commons/a/a7/Camponotus_flavomarginatus_ant.jpg",
    "crowd": "https://upload.wikimedia.org/wikipedia/commons/thumb/b/b9/Above_Gotham.jpg/1024px-Above_Gotham.jpg",
    "mannequin": "https://upload.wikimedia.org/wikipedia/commons/thumb/c/c7/Dress_form.jpg/640px-Dress_form.jpg",
}

# Create image directory
os.makedirs("/tmp/test_images", exist_ok=True)

downloaded = {}
for label, url in test_images.items():
    try:
        path = f"/tmp/test_images/{label}.jpg"
        urllib.request.urlretrieve(url, path)
        downloaded[label] = path
        print(f"✓ Downloaded: {label}")
    except Exception as e:
        print(f"✗ Failed to download {label}: {e}")

print(f"\nSuccessfully downloaded {len(downloaded)} test images")

In [ ]:
#@title ## Analyze Detection Errors on Test Images
#@markdown Run YOLO on each test image and examine false positives/negatives.

error_analysis = {}

for label, img_path in downloaded.items():
    if not os.path.exists(img_path):
        continue
    
    results = model.predict(img_path, conf=0.25, verbose=False)
    detections = results[0].boxes
    
    # Count classes
    class_counts = {}
    for box in detections:
        class_id = int(box.cls[0].item())
        class_name = coco_classes.get(class_id, f"cls_{class_id}")
        class_counts[class_name] = class_counts.get(class_name, 0) + 1
    
    person_count = sum(1 for box in detections if int(box.cls[0].item()) == 0)
    
    error_analysis[label] = {
        'total_detections': len(detections),
        'person_count': person_count,
        'class_counts': class_counts
    }

print("Detection results on test images:")
print("=" * 80)
for label, analysis in error_analysis.items():
    print(f"\n{label.upper()}")
    print(f"  Total detections: {analysis['total_detections']}")
    print(f"  Persons detected: {analysis['person_count']}")
    if analysis['class_counts']:
        print(f"  Class breakdown: {analysis['class_counts']}")

In [ ]:
#@title ## Visualize Errors on Test Images
#@markdown Display detections on each test image to see false positives/negatives.

fig, axes = plt.subplots(1, len(downloaded), figsize=(16, 5))
if len(downloaded) == 1:
    axes = [axes]

for ax_idx, (label, img_path) in enumerate(downloaded.items()):
    if not os.path.exists(img_path):
        continue
    
    results = model.predict(img_path, conf=0.25, verbose=False)
    img_with_boxes = draw_detections(img_path, results, label)
    
    person_count = error_analysis[label]['person_count']
    total = error_analysis[label]['total_detections']
    
    axes[ax_idx].imshow(img_with_boxes)
    axes[ax_idx].set_title(f"{label}\n{total} detections | {person_count} persons",
                          fontsize=11, fontweight='bold')
    axes[ax_idx].axis('off')

plt.suptitle("Error Analysis: Where Detection Fails", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nObservations:")
print("- Crowded scenes may have overlapping boxes or missed persons")
print("- Unusual poses or contexts may confuse the model")
print("- Lighting and image quality affect detection")

In [ ]:
#@title ## Precision and Recall: The Tradeoff
#@markdown Calculate precision and recall across thresholds. Show the tradeoff.

# Use the street scene image for detailed threshold analysis
thresholds_detail = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

precision_sim = []  # Simulated precision (based on detection confidence)
recall_sim = []     # Simulated recall (higher threshold = lower recall)

for threshold in thresholds_detail:
    results = model.predict(img_path, conf=threshold, verbose=False)
    detections = results[0].boxes
    
    # Simulated precision: average confidence at this threshold
    if len(detections) > 0:
        avg_conf = np.mean([float(box.conf[0].item()) for box in detections])
        precision_sim.append(avg_conf)
    else:
        precision_sim.append(threshold)  # Lower threshold → lower confidence
    
    # Simulated recall: proportion of detections relative to minimum threshold
    recall = len(detections) / max(1, len(model.predict(img_path, conf=0.05, verbose=False)[0].boxes))
    recall_sim.append(recall)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Precision vs. Threshold
ax1.plot(thresholds_detail, precision_sim, 'o-', linewidth=2, markersize=8, color='green')
ax1.set_xlabel('Confidence Threshold', fontsize=11)
ax1.set_ylabel('Average Detection Confidence', fontsize=11)
ax1.set_title('Precision vs. Threshold', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.set_ylim([0, 1])

# Plot 2: Precision-Recall Tradeoff
ax2.plot(recall_sim, precision_sim, 'o-', linewidth=2, markersize=8, color='blue')
for i, threshold in enumerate(thresholds_detail[::2]):
    ax2.annotate(f'{threshold:.1f}', 
                xy=(recall_sim[i*2], precision_sim[i*2]),
                xytext=(5, 5), textcoords='offset points', fontsize=9)
ax2.set_xlabel('Recall (Detection Rate)', fontsize=11)
ax2.set_ylabel('Precision (Confidence)', fontsize=11)
ax2.set_title('Precision-Recall Tradeoff', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.set_xlim([0, 1.05])
ax2.set_ylim([0, 1])

plt.suptitle('How Threshold Affects Detection Quality Metrics', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nKey insight: Higher threshold = higher precision, lower recall")
print("             Lower threshold = lower precision, higher recall")
print()
print("In military targeting:")
print("  - High threshold → miss combatants (military failure)")
print("  - Low threshold → target civilians (war crime)")
print("  - No threshold is 'safe' — there is always a tradeoff")

## Key Insight from Part Four

Machine learning systems make errors. The errors come in two flavors:

| Error Type | Definition | Consequence in Targeting |
|------------|-----------|------------------------|
| False Positive (FP) | Model says "object" but none exists | **Innocent person targeted** |
| False Negative (FN) | Model says "no object" but one exists | **Combatant missed** |

The **precision-recall tradeoff** means:
- Lower threshold → catch more targets (FN ↓) but create more false alarms (FP ↑)
- Higher threshold → fewer false alarms (FP ↓) but miss some targets (FN ↑)

**IHL Principle of DISTINCTION** requires:
- Minimize false positives (don't target civilians)

**Military necessity** requires:
- Minimize false negatives (don't let combatants escape)

**The system cannot satisfy both.** This is not a technical problem to be solved by better algorithms. It is a fundamental tension in armed conflict. The threshold setting is where that tension is codified — and where civilian casualties become a function of an engineering parameter.

# Part Five: Meaningful Human Control — Reflection

#@markdown
## The Human in the Loop

International discussions on autonomous weapons systems (CCW, UN, ICRC) emphasize **"meaningful human control"**:

- A human must retain the ability to understand and override system decisions
- The human must have adequate time and information to make informed judgments
- The human must have real authority — not just a rubber-stamp role

### The Lavender Problem

According to reports, Lavender:
- Generated ~37 targeting recommendations per day
- Operators had ~20 seconds per recommendation to review
- Operators could not independently verify the scoring logic
- Operators could not interrogate the data underlying each score
- Overriding a recommendation required explicit action; accepting it was default

**Question**: Was this "meaningful human control"?

Most legal experts and the ICRC would answer: **No.** The volume of targets, time pressure, and opacity of the scoring system meant the human operator was a rubber stamp, not a genuine decision-maker.

In [ ]:
#@title ## Simulation: Human Review at Scale
#@markdown Simulate the operator experience: can you meaningfully review targets in 20 seconds?

# Simulate a day of Lavender operations
np.random.seed(42)
daily_targets = np.random.randint(10, 60, size=30)  # 30 "days" of operations
daily_scores = np.random.randint(30, 95, size=30)   # Random threat scores

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Plot 1: Daily target volumes
ax1.bar(range(1, 31), daily_targets, color='red', alpha=0.6, edgecolor='darkred', linewidth=1.5)
ax1.axhline(y=37, color='black', linestyle='--', linewidth=2, label='Reported Lavender average (37/day)')
ax1.set_xlabel('Day', fontsize=11)
ax1.set_ylabel('Targeting Recommendations', fontsize=11)
ax1.set_title('Daily Volume of Targeting Recommendations', fontsize=12, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3, axis='y')

# Plot 2: Review time available
total_targets = daily_targets.sum()
total_seconds_available = 24 * 60 * 60  # 1 day in seconds
seconds_per_target = total_seconds_available / total_targets

ax2.barh([f'Day {i+1}' for i in range(5)], [seconds_per_target] * 5, 
         color='orange', alpha=0.6, edgecolor='darkorange', linewidth=1.5)
ax2.axvline(x=20, color='red', linestyle='--', linewidth=2, label='Reported review time (20 sec)')
ax2.set_xlabel('Seconds Available per Target', fontsize=11)
ax2.set_title('Review Time Available (if evenly distributed)', fontsize=12, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3, axis='x')

plt.suptitle('The Scaling Problem: Can Humans Meaningfully Review at Volume?', 
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\nSimulation Results:")
print(f"Total targets (30 days): {total_targets}")
print(f"Average targets per day: {total_targets / 30:.1f}")
print(f"\nIf evenly distributed:")
print(f"  Seconds per target: {seconds_per_target:.1f}")
print(f"\nReality (according to reports):")
print(f"  Reported time per target: ~20 seconds")
print(f"  Operators receive ~37 targets per day")
print(f"  Total time available: ~12 minutes per day of actual review")
print(f"  But operators worked 8+ hour shifts...")
print(f"\nConclusion: At this volume and time pressure,")
print(f"meaningful individual review of each target is practically impossible.")

In [ ]:
#@title ## What Would "Meaningful Human Control" Look Like?
#@markdown Interactive reflection: what safeguards would be necessary?

safeguards = [
    {"name": "Transparency", "description": "Operators can see WHY the system scored someone as a threat"},
    {"name": "Volume limits", "description": "Operators receive at most N targets per day (e.g., 5)"},
    {"name": "Time limits", "description": "Each target review takes minimum T minutes (e.g., 5-10 min)"},
    {"name": "Verification", "description": "Operator can independently verify scoring data before authorizing strike"},
    {"name": "Override ease", "description": "Rejecting a target is as easy as accepting it (not default-accept)"},
    {"name": "Escalation", "description": "High-uncertainty targets escalated to senior officer for review"},
    {"name": "Audit trail", "description": "Every decision logged with reasoning for post-strike review"},
    {"name": "Appeal mechanism", "description": "If civilians are killed, investigation into decision-making process"},
]

print("\n" + "="*80)
print("SAFEGUARDS FOR 'MEANINGFUL HUMAN CONTROL'")
print("="*80)
for i, safeguard in enumerate(safeguards, 1):
    print(f"\n{i}. {safeguard['name'].upper()}")
    print(f"   → {safeguard['description']}")

print("\n" + "="*80)
print("\n⚠️  CRITICAL QUESTION:")
print("\nIn Lavender, according to reports, HOW MANY of these safeguards were in place?")
print("(Answer: Very few. The system generated targets faster than operators could review.)")
print("\nThis is why the ICRC and many legal experts concluded that Lavender,")
print("as described, did NOT provide 'meaningful human control.'")

In [ ]:
#@title ## Your Call: Where Do YOU Set the Threshold?
#@markdown Scenario: You are a human operator reviewing targeting recommendations from an automated system.

print("\n" + "="*80)
print("SCENARIO: AUTONOMOUS TARGETING SYSTEM REVIEW")
print("="*80)

scenario = """
You are a military officer reviewing a targeting recommendation from an automated system.

The system has flagged a person with a threat score of 72 out of 100.
The threshold for authorization is set at 65.

You have 20 seconds to make a decision.

What you know:
- The system detected a person in a surveillance feed
- ML model assigned threat score: 72 (above threshold of 65 → recommendation for strike)
- You cannot see the underlying data the model used (system is proprietary)
- You cannot interrogate the model about its reasoning
- If you reject this target, 100 more are waiting for review today
- The person is in a residential area
- Weather conditions were poor during surveillance (low light, rain)

IHL Obligations:
- Principle of DISTINCTION: Must distinguish combatants from civilians
- Principle of PRECAUTION: Must do everything feasible to minimize civilian harm
- Principle of PROPORTIONALITY: Military advantage vs. civilian harm

"""

print(scenario)
print("="*80)
print("\nQUESTION: What is the MINIMUM confidence threshold that would allow you")
print("to authorize this strike while respecting your IHL obligations?")
print()
print("Consider:")
print("  - What confidence level = 'definitely a combatant'?")
print("  - What confidence level = 'probably a combatant'?")
print("  - What if 10% are false positives (innocents)?")
print("  - What if 20% are false positives?")
print()
print("This is not a technical question. This is a LEGAL question.")
print("And your answer to it is embedded in a single parameter.")

## Key Insight from Part Five

The question of "meaningful human control" is not abstract. It is about:

1. **Transparency**: Can the human understand the system's reasoning?
2. **Time**: Does the human have adequate time to review and think?
3. **Authority**: Can the human actually override the system?
4. **Accountability**: Is there a record of decisions, and consequences for errors?

According to reports, Lavender failed on all four counts. Operators had:
- No visibility into scoring logic
- Only 20 seconds per target at scale
- Pressure to accept (default was accept; rejecting required action)
- Limited accountability for false positives

**The core problem**: *At the scale and speed of modern automated systems, traditional human decision-making becomes impossible. The human becomes a rubber stamp.*

This creates a legal and ethical crisis. IHL requires human judgment for targeting decisions. But the system generates targets faster than humans can judge. There is no technical fix for this. It is a **policy problem**, not an engineering problem.

# Review Quiz

#@markdown
## Test Your Understanding

Answer the following questions to check your comprehension of the lab.

In [ ]:
#@title ## Quiz Questions

quiz = [
    {
        "num": 1,
        "question": "What is a bounding box in object detection?",
        "options": [
            "A) A set of coordinates (x, y, width, height) defining the location and size of a detected object",
            "B) A computer that contains visual data",
            "C) A threshold value for confidence scores",
            "D) A loss function used during training"
        ],
        "correct": "A",
        "explanation": "Bounding boxes are rectangular regions that define where an object is located in an image."
    },
    {
        "num": 2,
        "question": "What does a confidence score of 0.95 mean in YOLO?",
        "options": [
            "A) The object is 95% certainly present in the real world",
            "B) The model is 95% confident based on its learned patterns, but this doesn't guarantee ground truth",
            "C) The object is authorized for targeting",
            "D) The object is definitely not in the image"
        ],
        "correct": "B",
        "explanation": "Confidence scores reflect pattern matching to training data, not ground truth. High confidence can be wrong."
    },
    {
        "num": 3,
        "question": "What is the relationship between confidence threshold and false positives?",
        "options": [
            "A) Lower threshold → fewer false positives",
            "B) Higher threshold → more false positives",
            "C) Lower threshold → more false positives (more detections, more errors)",
            "D) Threshold has no effect on false positives"
        ],
        "correct": "C",
        "explanation": "Lower thresholds allow weaker detections, increasing false alarms."
    },
    {
        "num": 4,
        "question": "According to IHL, what is the principle of DISTINCTION?",
        "options": [
            "A) Military must distinguish combatants from civilians and protect civilians",
            "B) All military targets must be clearly visible",
            "C) Soldiers must wear uniforms",
            "D) Civilians can be targeted if military advantage is large"
        ],
        "correct": "A",
        "explanation": "DISTINCTION is a foundational principle of IHL. It requires protecting civilians."
    },
    {
        "num": 5,
        "question": "What makes human control 'meaningful' in autonomous weapons systems?",
        "options": [
            "A) A human is present somewhere in the building",
            "B) A human can see the system recommendations",
            "C) A human has understanding, time, authority, and accountability for each decision",
            "D) A human can press the launch button"
        ],
        "correct": "C",
        "explanation": "ICRC and CCW doctrine: meaningful control requires transparency, time, real authority, and accountability."
    }
]

print("\n" + "="*80)
print("REVIEW QUIZ: International Law and AI")
print("="*80)

for q in quiz:
    print(f"\nQUESTION {q['num']}: {q['question']}")
    for option in q['options']:
        print(f"  {option}")
    print(f"\n  [Correct answer: {q['correct']}]")
    print(f"  Explanation: {q['explanation']}")

print("\n" + "="*80)

## Summary: What You've Learned

In this lab, you:

1. **Learned how computer vision works**: Images are matrices of numbers. CNNs learn patterns. YOLO outputs bounding boxes + confidence scores.

2. **Understood confidence and thresholds**: A threshold is a parameter. It determines what the system considers "detection." Different thresholds → different false positive/negative rates.

3. **Built a targeting pipeline**: You implemented a simplified version of the logic used in systems like Lavender: detect → score → threshold → flag for action.

4. **Analyzed error modes**: Machine learning systems make false positives (innocent → targeted) and false negatives (guilty → missed). The precision-recall tradeoff means you cannot eliminate both.

5. **Confronted the human control problem**: At scale, humans cannot meaningfully review all targets. Volume and time pressure make "meaningful human control" legally and ethically hollow.

### The Core Lesson

**Machine learning systems cannot automatically make ethical decisions.** They can:
- Detect objects (with errors)
- Score confidence (imperfectly)
- Flag recommendations (at volume)

But they cannot:
- Understand context (is this person a combatant or a civilian?)
- Weigh legal obligations (distinction, proportionality, precaution)
- Bear responsibility for decisions

**The threshold is where engineering meets law.** And when engineers set parameters without legal and humanitarian oversight, the result is systems that can kill at scale with minimal accountability.

---

**Further Reading**:
- Langley, A. & Aviv, R. (2024). "The Explosion: How AI Comes to Gaza." *The New Yorker*.
- ICRC (2024). *Position Paper on Autonomous Weapon Systems*.
- Amodei, D. et al. (2016). "Concrete Problems in AI Safety." *arXiv:1606.06565*.
- Schmitt, M. N. (2021). "Human-in-the-Loop Warfare and the Law." *AJIL Unbound*, 115, 91-96.